In [1]:
import vk_api
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# Чтобы получить API: https://oauth.vk.com/authorize?client_id=<ID приложения>&display=page&redirect_uri=https://oauth.vk.com/blank.html&scope=wall,offline&response_type=token&v=5.199
vk = vk_api.VkApi(token=os.environ.get("VK_API"),
                  api_version="5.199").get_api()
group_data = vk.groups.getById(
    group_id="club237241770", fields="description")["groups"][0]
group_name = group_data["name"]
group_desc = group_data.get("description")

owner_id = -int(group_data["id"])
group_posts = vk.wall.get(owner_id=owner_id, count=100, extended=1)["items"]
group_pin_post = group_posts[0].get(
    "text") if group_posts[0].get("is_pinned") == 1 else ""

group_reg_posts = []
for post_id in range(group_posts[0].get("is_pinned") == 1, len(group_posts)):
    group_post_text = group_posts[post_id].get("text")
    if group_posts[post_id].get("geo"):
        group_post_geo_data = group_posts[post_id].get(
            "geo").get("coordinates")
        group_reg_posts.append([group_post_text, group_post_geo_data])
    else:
        group_reg_posts.append([group_post_text, ""])

print("Имя группы: ", group_name)
print("Описание группы:\n", group_desc)
print("Закрепленное сообщение группы:\n", group_pin_post)
print("Последние 99 - 100 сообщений:\n", group_reg_posts)

client = OpenAI(base_url=os.environ.get("OPENAI_API"),
                api_key=os.environ.get("OPENAI_API_KEY"))

relevant_posts = []
for post in group_reg_posts:
    is_trash_confirmed = ""
    if post[0] != "":
        is_trash_confirmed = client.chat.completions.create(
            model=os.environ.get("MODEL"),
            messages=[
                {"role": "system", "content": "Определи, описан ли в тексте конкретный случай, где мусор лежит у железнодорожных путей. Отвечай только 'да' или 'нет'."},
                {"role": "user", "content": post[0]}
            ]
        )
    is_trash_confirmed = True if is_trash_confirmed and is_trash_confirmed.choices[0].message.content.lower(
    ).strip() == "да" else False
    if is_trash_confirmed:
        relevant_posts.append(post)
        if relevant_posts[-1][1] == "":
            trash_geo = client.chat.completions.create(
                model=os.environ.get("MODEL"),
                messages=[
                    {"role": "system", "content": "В тексте описан конкретный случай, где мусор лежит у железнодорожных путей. Если в тексте указан конкретный адрес, где он лежит, то твой ответ должен быть этим адресом, иначе отвечай 'адреса нет'."},
                    {"role": "user", "content": post[0]}
                ]
            )
            if trash_geo.choices[0].message.content.lower() != "адреса нет":
                relevant_posts[-1][1] = trash_geo.choices[0].message.content.lower()


print("Релевантные посты:\n", relevant_posts)

Имя группы:  Testing
Описание группы:
 
Закрепленное сообщение группы:
 
Последние 99 - 100 сообщений:
 [['🌍ИНТЕРЕСНАЯ ШКОЛА🌍\n🔔Запись в первый класс открыта!\n\n🥇Первый в Дзержинском семейный класс полного дня с 09:00 до 16:00 в КИД! А с 16:00 до 19:00 - продлёнка.\n\n❤\u200d Счастье - когда ребёнка учат профессиональные, влюбленные в дело Педагоги.\n❤\u200d Когда делаешь домашку в школе, а не дома с родителями.\n❤\u200d Когда в классе уютно, весело и интересно, а к твоему мнению всегда прислушаются.\n☀Спокойствие для родителей, потрясающее пространство КИД для развития ребёнка - каждый день.\n🔥И всё это - рядом с домом!!\n\n🔝Подробности выше.\n❗️Презентация нашего невероятного проекта уже 7 марта!\n✏Только по предварительной записи!\n\n🌍Клуб Интересных Детей — Мир интереснее экрана телефона.\n\n✅Канал Telegram: t.me/Kid_Dzr\n⭐️⭐️⭐️⭐️⭐️\nКарточка Яндекс:\nyandex.ru/maps/org/k...\n\n🍀Мы рядом🍀\n🎯Лермонтова,21\n🔸8(995)919-88-38\n\n🔸Запись в наш класс:\n🔸8(915)387-07-03\n🔸Маша, руководит